In [19]:
import argparse
import os
import sys
import warnings
import numpy as np

warnings.filterwarnings("ignore", category=FutureWarning)

import pandas as pd
import torch
from sklearn.model_selection import KFold
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

In [2]:
%load_ext autoreload
%autoreload 2

from drGT import drGT
from drGT.load_data import load_data
from drGT.metrics import evaluate_predictions
from drGT.myutils import get_all_edges_and_labels, get_model_params
from drGT.sampler import BalancedSampler

In [3]:
method = "Transformer"
data = "nci"
task = "test1"
params = get_model_params(task, data, method)

In [4]:
# Load data
(
    drugAct,
    null_mask,
    S_d,
    S_c,
    S_g,
    _,
    _,
    _,
    A_cg,
    A_dg,
) = load_data(data, is_zero_pad=params["is_zero_pad"], verbose=True)

# Update parameters
params.update(
    {
        "n_drug": S_d.shape[0],
        "n_cell": S_c.shape[0],
        "n_gene": S_g.shape[0],
        "gnn_layer": method,
    }
)

load nci
unique drugs	976
unique cells	59
unique drug response	14312
n sensitive	7050
n resistant	7262
AVG Drug binary ratio	0.485
AVG Cell binary ratio	0.497
Over 10 entries (drugs)	853
Over 10 entries (cells)	59
dtis	572
unique drugs	191
unique genes	242
Top 90% variable genes: 	2247
Total selected genes: 	2489
Done!


In [6]:
all_edges, all_labels = get_all_edges_and_labels(drugAct, null_mask)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, test_idx in tqdm(kf.split(all_edges)):
    sampler = BalancedSampler(
        drugAct,
        all_edges,
        all_labels,
        train_idx,
        test_idx,
        null_mask,
        S_d,
        S_c,
        S_g,
        A_cg,
        A_dg,
    )

    break

0it [00:00, ?it/s]


In [6]:
probs, true_labels, attention = drGT.predict("best_model.pt", sampler, params)

Using device for prediction: cpu


In [7]:
evaluate_predictions(true_labels, probs)

,Score
Metric,
accuracy,0.8655
f1_score,0.8580
auroc,0.9411
aupr,0.9387


## For your specific cell lines and drugs

Here, `all_edges` and `all_labels` are constructed from the `drugAct` matrix, where each edge corresponds to a (cell line, drug) pair.  
If you want to focus on specific cell lines or drugs, you can define `test_idx` to include the corresponding edges and use the remaining edges as `train_idx`.

Note that this setup is typically used either for targeted evaluation (with an appropriate split to avoid data leakage) or for inference with a pretrained model, depending on your use case.

In [15]:
print(drugAct.shape)
drugAct.head()

(59, 976)


,740,752,755,757,762,1390,1895,3053,3061,3088,...,808790,808792,809693,810341,810717,811429,812926,812927,813488,820919
786_0,1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,1.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN
A498,0.0,0.0,0.0,0.0,NaN,0.0,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,1.0,0.0
A549,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ACHN,1.0,NaN,NaN,NaN,1.0,NaN,1.0,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,1.0,1.0,NaN,1.0,NaN
BT_549,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0


In [16]:
drugAct.index

Index(['786_0', 'A498', 'A549', 'ACHN', 'BT_549', 'CAKI_1', 'CCRF_CEM',
       'COLO205', 'DU_145', 'EKVX', 'HCC_2998', 'HCT_116', 'HCT_15', 'HL_60',
       'HOP_62', 'HOP_92', 'HS578T', 'HT29', 'IGROV1', 'KM12', 'K_562',
       'LOXIMVI', 'M14', 'MALME_3M', 'MCF7', 'MDA_MB_231', 'MDA_MB_435',
       'MDA_N', 'MOLT_4', 'NCI_ADR_RES', 'NCI_H226', 'NCI_H23', 'NCI_H322M',
       'NCI_H460', 'NCI_H522', 'OVCAR_3', 'OVCAR_4', 'OVCAR_5', 'OVCAR_8',
       'PC_3', 'RPMI_8226', 'RXF_393', 'SF_268', 'SF_295', 'SK_MEL_2',
       'SK_MEL_28', 'SK_MEL_5', 'SK_OV_3', 'SN12C', 'SNB_19', 'SNB_75', 'SR',
       'SW_620', 'T47D', 'TK_10', 'U251', 'UACC_257', 'UACC_62', 'UO_31'],
      dtype='object')

In [13]:
print(len(all_edges))
all_edges

14312


array([[  0,   0],
       [  0,   6],
       [  0,   9],
       ...,
       [ 58, 941],
       [ 58, 958],
       [ 58, 963]])

In [10]:
all_labels

array([1, 1, 1, ..., 0, 0, 0])

In [17]:
row_786 = drugAct.index.get_loc('786_0')
print(row_786)

0


In [22]:
test_idx = np.where(all_edges[:, 0] == row_786)[0]
train_idx = np.setdiff1d(train_idx, edges_786)

In [23]:
np.unique(all_edges[test_idx][:, 0])

array([0])

In [24]:
sampler = BalancedSampler(
    drugAct,
    all_edges,
    all_labels,
    train_idx,
    test_idx,
    null_mask,
    S_d,
    S_c,
    S_g,
    A_cg,
    A_dg,
)

In [25]:
probs, true_labels, attention = drGT.predict("best_model.pt", sampler, params)

Using device for prediction: cpu


In [26]:
evaluate_predictions(true_labels, probs)

,Score
Metric,
accuracy,0.8673
f1_score,0.9207
auroc,0.8917
aupr,0.9767
